# 7.30 — 3D vision & point clouds (PointNet)

PointNet respects a point cloud's odd truth: the shape is the set of points, not the order in which the file happened to list them.

3D point clouds represent surfaces as unordered samples in space. Shared point functions and symmetric pooling turn those samples into a global descriptor without depending on the file's row order. Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(730)
np.set_printoptions(precision=4, suppress=True)

## Build PointNet once

PointNet applies the same function to every point, then aggregates with a symmetric max:

$$g(P)=\max_{p_i\in P}h(p_i)$$

The max is coordinate-wise, so row order is not part of the geometry.

In [ ]:
def pointnet_embed(points, feature_fn=None):
    points = np.asarray(points, dtype=float)
    if feature_fn is None:
        feature_fn = lambda p: np.array([p[0] + 2.0 * p[1], p[2]])
    features = np.array([feature_fn(point) for point in points])
    pooled = features.max(axis=0)
    return features, pooled

cloud = np.array([
    [0, 0, 0],
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1],
], dtype=float)
feature_table = np.array([
    [1, 3],
    [2, 1],
    [0, 5],
], dtype=float)
pooled = feature_table.max(axis=0)
reordered = feature_table[[2, 0, 1]]
pooled_reordered = reordered.max(axis=0)
print("cloud shape", cloud.shape)
print("pooled", pooled)
print("reordered pooled", pooled_reordered)
assert cloud.shape == (4, 3)
assert np.allclose(pooled, [2, 5])
assert np.allclose(pooled_reordered, [2, 5])

The shared function example maps $[1,0,0]$ to $[1,0]$ and $[0,1,0]$ to $[2,0]$. Duplicating the strongest feature leaves the max unchanged, which is a useful robustness property and a counting pitfall.

In [ ]:
features_two, pooled_two = pointnet_embed(np.array([[1, 0, 0], [0, 1, 0]], dtype=float))
duplicate_a = np.array([[2, 5], [1, 3]], dtype=float)
duplicate_b = np.array([[2, 5], [2, 5], [1, 3]], dtype=float)
print("shared features", features_two)
print("two-point max", pooled_two)
print("duplicate max", duplicate_a.max(axis=0), duplicate_b.max(axis=0))
assert np.allclose(features_two, [[1, 0], [2, 0]])
assert np.allclose(pooled_two, [2, 0])
assert np.allclose(duplicate_a.max(axis=0), [2, 5])
assert np.allclose(duplicate_b.max(axis=0), [2, 5])

## Synthetic point-cloud ladder

Each rung has two classes: elongated bar-like clouds and compact shell-like clouds. Difficulty rises through more points, rotation, noise, outliers, missing points, and duplicated samples.

In [ ]:
def sample_bar(n, noise, seed):
    local = np.random.default_rng(seed)
    x = local.uniform(-1.0, 1.0, n)
    y = local.normal(0.0, noise, n)
    z = local.normal(0.0, noise, n)
    return np.column_stack([x, y, z])


def sample_shell(n, noise, seed):
    local = np.random.default_rng(seed)
    angles = local.uniform(0.0, 2.0 * np.pi, n)
    radius = 0.55 + local.normal(0.0, noise, n)
    x = radius * np.cos(angles)
    y = radius * np.sin(angles)
    z = local.normal(0.0, noise, n)
    return np.column_stack([x, y, z])


def rotate_z(points, angle):
    c = np.cos(angle)
    s = np.sin(angle)
    matrix = np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]])
    return points @ matrix.T


def corrupt(points, outliers, drop_rate, duplicates, seed):
    local = np.random.default_rng(seed)
    keep = local.random(len(points)) > drop_rate
    points = points[keep]
    if duplicates > 0 and len(points) > 0:
        picks = local.integers(0, len(points), duplicates)
        points = np.vstack([points, points[picks]])
    if outliers > 0:
        extras = local.uniform(-1.2, 1.2, size=(outliers, 3))
        points = np.vstack([points, extras])
    return points


def normalize_cloud(points):
    centered = points - points.mean(axis=0, keepdims=True)
    scale = np.linalg.norm(centered, axis=1).max() + 1e-9
    return centered / scale


def build_cloud_rung(name, n_points, n_each, noise, rotation, outliers, drop_rate, duplicates, seed):
    clouds = []
    labels = []
    for i in range(n_each):
        angle = rotation * ((i % 7) / 6.0)
        bar = rotate_z(sample_bar(n_points, noise, seed + i), angle)
        shell = rotate_z(sample_shell(n_points, noise, seed + 1000 + i), -angle)
        clouds.append(normalize_cloud(corrupt(bar, outliers, drop_rate, duplicates, seed + 2000 + i)))
        labels.append(0)
        clouds.append(normalize_cloud(corrupt(shell, outliers, drop_rate, duplicates, seed + 3000 + i)))
        labels.append(1)
    return name, clouds, np.array(labels)


def shape_embed(points):
    points = np.asarray(points, dtype=float)
    radii = np.linalg.norm(points, axis=1)
    features = np.column_stack([
        np.abs(points[:, 0]),
        np.abs(points[:, 1]),
        np.abs(points[:, 2]),
        radii,
        -radii,
    ])
    pooled = features.max(axis=0)
    spread = points.std(axis=0)
    return np.concatenate([pooled, spread])


def nearest_centroid_accuracy(clouds, labels):
    embeddings = np.array([shape_embed(points) for points in clouds])
    train_mask = np.arange(len(labels)) % 3 != 0
    test_mask = ~train_mask
    centroids = []
    for cls in sorted(set(labels.tolist())):
        centroids.append(embeddings[train_mask & (labels == cls)].mean(axis=0))
    centroids = np.array(centroids)
    distances = ((embeddings[test_mask, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
    preds = distances.argmin(axis=1)
    return float((preds == labels[test_mask]).mean())

rungs = [
    build_cloud_rung("D1 four-point toy", 16, 18, 0.01, 0.00, 0, 0.00, 0, 1),
    build_cloud_rung("D2 more samples", 32, 24, 0.03, 0.20, 0, 0.00, 1, 2),
    build_cloud_rung("D3 rotated noisy", 48, 30, 0.06, 0.80, 2, 0.05, 2, 3),
    build_cloud_rung("D4 missing outliers", 64, 36, 0.09, 1.30, 6, 0.15, 4, 4),
    build_cloud_rung("D5 dense corrupted", 80, 42, 0.13, 2.20, 12, 0.25, 10, 5),
]
for name, clouds, labels in rungs:
    lengths = [len(points) for points in clouds[:4]]
    print(f"{name:22s} clouds={len(clouds)} example_points={lengths} classes={sorted(set(labels.tolist()))}")

In [ ]:
metrics = []
for name, clouds, labels in rungs:
    acc = nearest_centroid_accuracy(clouds, labels)
    metrics.append(acc)
    print(f"{name:22s} accuracy={acc:.3f}")

In [ ]:
fig = plt.figure(figsize=(14, 5))
for idx, (name, clouds, labels) in enumerate(rungs):
    ax = fig.add_subplot(2, 5, idx + 1, projection="3d")
    pts = clouds[0]
    ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=8)
    ax.set_title(name.split()[0])
    ax.set_axis_off()
ax = fig.add_subplot(2, 1, 2)
ax.plot(range(1, 6), metrics, marker="o")
ax.set_ylim(0, 1.05)
ax.set_xlabel("rung")
ax.set_ylabel("accuracy")
ax.set_title("PointNet-style embedding vs difficulty")
fig.tight_layout()

## Pitfall on D5: feeding point order into the model

A bad baseline that reads the first few rows treats file order as geometry. Shuffling the same cloud changes its embedding. The PointNet-style max embedding stays invariant.

In [ ]:
def bad_order_embed(points):
    padded = np.zeros((8, 3))
    take = min(8, len(points))
    padded[:take] = points[:take]
    return padded.ravel()

name, clouds, labels = rungs[-1]
points = clouds[0]
shuffled = points[np.random.default_rng(9).permutation(len(points))]
bad_change = np.linalg.norm(bad_order_embed(points) - bad_order_embed(shuffled))
good_change = np.linalg.norm(shape_embed(points) - shape_embed(shuffled))
print("bad order-sensitive change", round(float(bad_change), 4))
print("PointNet-style change", round(float(good_change), 4))
assert bad_change > 0.0
assert np.isclose(good_change, 0.0)

## Evaluate it + practice

- Track accuracy against a majority-class baseline.
- Sanity check: shuffle every cloud; a PointNet-style metric should not move.
- Ablation: replace max pooling with row-index features; metric becomes order-sensitive.
- Failure signals: poor rotation handling, outlier sensitivity, or density/count information disappearing.

Practice: add a third class shaped like a vertical column.

In [ ]:
# Your experiment here

Practice: compare max pooling with mean pooling on duplicated points.

In [ ]:
# Your experiment here

Practice: normalize clouds with and without scale division and compare embeddings.

In [ ]:
# Your experiment here